# RAG: Baseline Dense Retrieval vs. Cross-Encoder Re-Ranking
Evaluating two-stage retrieval on the Catan rulebook using `all-MiniLM-L6-v2`, `ms-marco-MiniLM-L-6-v2`, and Gemini.


## 1. Setup

In [2]:
!pip install -q google-genai pypdf sentence-transformers faiss-cpu pandas requests

In [3]:
import os, re, time, getpass
import numpy as np
import pandas as pd
import faiss
import requests
from IPython.display import display
from pypdf import PdfReader
from sentence_transformers import SentenceTransformer, CrossEncoder
from google import genai
from google.genai import types

pd.set_option("display.max_colwidth", 90)
pd.set_option("display.width", 200)

# Configuration
PDF_URL = "https://www.catan.com/sites/prod/files/2021-06/catan_base_rules_2020_200707.pdf"
PDF_PATH = "catan_rules.pdf"

GEN_MODEL = "gemini-3.8-flash"
FALLBACK_MODELS = ["gemini-3.8-flash", "gemini-flash-latest", "gemini-3.5-flash", "gemini-3.6-flash"]
LOCAL_EMBED_MODEL = "sentence-transformers/all-MiniLM-L6-v2"
CROSS_ENCODER_NAME = "cross-encoder/ms-marco-MiniLM-L-6-v2"

CHUNK_WORDS = 180
OVERLAP_WORDS = 40
TOP_K_RETRIEVE = 10
TOP_K_FINAL = 3


W1019 20:14:35.349000 34356 site-packages/torch/distributed/elastic/multiprocessing/redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.


Dense vector search is run locally using `all-MiniLM-L6-v2` with FAISS, and Gemini (`gemini-3.8-flash`) is used for generation.


In [4]:
# API key setup
if not os.environ.get("GEMINI_API_KEY") and os.path.exists(".env"):
    for line in open(".env"):
        if line.startswith("GEMINI_API_KEY="):
            os.environ["GEMINI_API_KEY"] = line.strip().split("=", 1)[1]

API_KEY = os.environ.get("GEMINI_API_KEY")
if not API_KEY:
    try:
        import importlib
        userdata = importlib.import_module("google.colab.userdata")
        API_KEY = userdata.get("GEMINI_API_KEY")
    except Exception:
        API_KEY = getpass.getpass("Paste your Gemini API key: ")

client = genai.Client(api_key=API_KEY)


## 2. Load the PDF and chunk it

In [5]:
if not os.path.exists(PDF_PATH):
    try:
        r = requests.get(PDF_URL, timeout=60, headers={"User-Agent": "Mozilla/5.0"})
        r.raise_for_status()
        with open(PDF_PATH, "wb") as f:
            f.write(r.content)
    except Exception as e:
        try:
            r = requests.get("https://web.archive.org/web/20220308000000/https://www.catan.com/sites/prod/files/2021-06/catan_base_rules_2020_200707.pdf", timeout=60, headers={"User-Agent": "Mozilla/5.0"})
            r.raise_for_status()
            with open(PDF_PATH, "wb") as f:
                f.write(r.content)
        except Exception:
            raise FileNotFoundError(f"Please provide {PDF_PATH}")

reader = PdfReader(PDF_PATH)
pages = [(i + 1, p.extract_text() or "") for i, p in enumerate(reader.pages)]
print("pages:", len(pages), "| total characters:", sum(len(t) for _, t in pages))
print("\n--- sample of raw page 3 ---\n", pages[min(2, len(pages) - 1)][1][:600])


pages: 16 | total characters: 55189

--- sample of raw page 3 ---
 33
Starting Set-up for Experienced Players
It is more fun to play with a variable game board—with the 
game board laid out randomly. The board changes each game. 
If you would like to use the variable set-up, you can find the 
guidelines in the Almanac under Set-up, Variable Y. Also look 
for useful tips under Set-up Phase Y and Tactics Y.
settin G up the Game
Select a color and take your 5 settlements, 4 cities, and 
15 roads (no more and no less!). Place your 2 roads and 
your 2 settlements on the game board. Place your remaining 
settlements, roads, and cities down in front of you.
Note: If


Text extraction and sliding-window chunking (180 words, 40 words overlap).


In [6]:
def clean(text):
    text = re.sub(r"-\n(\w)", r"\1", text)
    text = re.sub(r"\s*\n\s*", " ", text)
    return re.sub(r"\s{2,}", " ", text).strip()

words = []
for pg, txt in pages:
    for w in clean(txt).split():
        words.append((w, pg))

chunks = []
step = CHUNK_WORDS - OVERLAP_WORDS
for start in range(0, len(words), step):
    window = words[start:start + CHUNK_WORDS]
    if start > 0 and len(window) <= OVERLAP_WORDS:
        break
    chunks.append({
        "id": len(chunks),
        "page": window[0][1],
        "text": " ".join(w for w, _ in window)
    })

print("total words:", len(words), "| chunks:", len(chunks))
for c in chunks[:2]:
    print(f"\n[chunk {c['id']} | page {c['page']}]\n{c['text'][:400]}...")


total words: 10121 | chunks: 73

[chunk 0 | page 1]
Klaus Teuber Game RulesGame Rules && almanac almanac 22 Dear Settlers, To make it as easy as possible for you to start playing Catan®, we use an award-winning rules system, which consists of 3 parts. First, if you do not know how to play Catan, please read the Game Overview on page 16 (the back cover). Next, read the Game Rules on pages 2-6 (red borders) and start to play the game. If you have que...

[chunk 1 | page 2]
the game, you can look up keywords (marked Y) in the “Almanac,” which follows these rules. Game Components • 19 terrain hexes (tiles) • 6 sea frame pieces • 9 harbor pieces • 18 circular number tokens • 95 resource cards (bearing the symbols for the brick, grain, lumber , ore, and wool resources) • 25 development cards (14 knight cards, 6 progress cards, 5 victory point cards) • 4 “Building Costs”...


## 3. Bi-Encoder Indexing (FAISS)


In [7]:
bi_encoder = SentenceTransformer(LOCAL_EMBED_MODEL)

def embed_docs(texts):
    return bi_encoder.encode(texts, batch_size=32, normalize_embeddings=True, show_progress_bar=True).astype("float32")

def embed_query(q):
    return bi_encoder.encode([q], normalize_embeddings=True).astype("float32")

t0 = time.perf_counter()
doc_vecs = embed_docs([c["text"] for c in chunks])
index_build_time = time.perf_counter() - t0

index = faiss.IndexFlatIP(doc_vecs.shape[1])
index.add(doc_vecs)
print(f"indexed {index.ntotal} chunks, dim={doc_vecs.shape[1]}, indexing time: {index_build_time:.2f}s")


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Batches:   0%|          | 0/3 [00:00<?, ?it/s]

indexed 73 chunks, dim=384, indexing time: 0.65s


## 4. The Two Pipelines
Pipeline 1 retrieves top-3 candidates via cosine similarity. Pipeline 2 re-ranks the initial top-10 candidates with the Cross-Encoder.


In [8]:
cross_encoder = CrossEncoder(CROSS_ENCODER_NAME)

def baseline_retrieve(query, k=TOP_K_RETRIEVE):
    scores, ids = index.search(embed_query(query), k)
    return [(int(i), float(s)) for i, s in zip(ids[0], scores[0])]

def rerank(query, candidates):
    pairs = [(query, chunks[cid]["text"]) for cid, _ in candidates]
    scores = cross_encoder.predict(pairs)
    scored = [(cid, float(sc)) for (cid, _), sc in zip(candidates, scores)]
    return sorted(scored, key=lambda x: x[1], reverse=True)


Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

In [9]:
PROMPT = """You are a rules referee for the board game Catan. Answer the question using ONLY the rulebook excerpts below.
If the excerpts do not contain the answer, say exactly: "The provided excerpts don't cover this."
Keep it short (max 4 sentences) and mention any exception or restriction that applies.

EXCERPTS:
{context}

QUESTION: {question}

ANSWER:"""

def ask_llm(question, chunk_ids):
    excerpts = []
    for n, cid in enumerate(chunk_ids):
        pg = chunks[cid]["page"]
        txt = chunks[cid]["text"]
        excerpts.append(f"[Excerpt {n + 1} | page {pg}]\n{txt}")
    context = "\n\n".join(excerpts)
    prompt = PROMPT.format(context=context, question=question)

    models_to_try = [GEN_MODEL] + [m for m in FALLBACK_MODELS if m != GEN_MODEL]
    for model_name in models_to_try:
        for attempt in range(2):
            try:
                t0 = time.perf_counter()
                resp = client.models.generate_content(
                    model=model_name, contents=prompt,
                    config=types.GenerateContentConfig(temperature=0))
                time.sleep(1)
                return (resp.text or "").strip(), time.perf_counter() - t0
            except Exception as e:
                err_str = str(e)
                if "429" in err_str or "RESOURCE_EXHAUSTED" in err_str:
                    break
                wait = 2 * (attempt + 1)
                time.sleep(wait)
    return "(LLM call failed)", float("nan")


## 5. Evaluation Helpers


In [10]:
RESULTS = []

GOLD_IDS = {
    "trade_negation":     [10],
    "road_broken":        [33],
    "bigger_settlement":  [13],
    "bank_trade_synonym": [34],
    "win_other_turn":     [26],
    "knight_army":        [16],
}

def find_chunks(pattern):
    return [c["id"] for c in chunks if re.search(pattern, c["text"], re.I)]

def gold_rank(ordered_ids, gold_ids):
    for r, cid in enumerate(ordered_ids, 1):
        if cid in gold_ids:
            return r
    return None

def compare(query, gold_ids=None, call_llm=True, tag="", **kwargs):
    print("=" * 110)
    print("QUERY:", query)

    t0 = time.perf_counter()
    cands = baseline_retrieve(query)
    t_base = time.perf_counter() - t0

    t0 = time.perf_counter()
    ranked = rerank(query, cands)
    t_rerank = time.perf_counter() - t0

    base_ids = [cid for cid, _ in cands]
    rr_ids = [cid for cid, _ in ranked]
    cos_map, ce_map = dict(cands), dict(ranked)

    rows = []
    for b_rank, cid in enumerate(base_ids, 1):
        rr_rank = rr_ids.index(cid) + 1
        rows.append({
            "chunk": cid, "page": chunks[cid]["page"],
            "base_rank": b_rank, "cosine": round(cos_map[cid], 3),
            "ce_score": round(ce_map[cid], 2), "rerank_rank": rr_rank,
            "moved": b_rank - rr_rank,
            "gold": "YES" if gold_ids and cid in gold_ids else "",
            "preview": chunks[cid]["text"][:85] + "..."
        })
    display(pd.DataFrame(rows).sort_values("rerank_rank").reset_index(drop=True))

    gb = gold_rank(base_ids, gold_ids) if gold_ids else None
    gr = gold_rank(rr_ids, gold_ids) if gold_ids else None
    if gold_ids:
        print(f"GOLD chunk rank:  baseline #{gb}  ->  re-ranked #{gr}   (None = not in the top-10 at all)")

    a1 = a2 = None
    t1 = t2 = float("nan")
    if call_llm:
        a1, t1 = ask_llm(query, base_ids[:TOP_K_FINAL])
        if base_ids[:TOP_K_FINAL] == rr_ids[:TOP_K_FINAL]:
            a2, t2 = a1, t1
            print("\n(top-3 identical for both pipelines, so the answers are the same)")
        else:
            a2, t2 = ask_llm(query, rr_ids[:TOP_K_FINAL])
        print("\nANSWER 1 (baseline):\n", a1)
        print("\nANSWER 2 (re-ranked):\n", a2)

    res = {
        "tag": tag, "query": query, "gold_base": gb, "gold_rerank": gr,
        "t_base": t_base, "t_rerank": t_rerank, "t_llm1": t1, "t_llm2": t2,
        "answer1": a1, "answer2": a2
    }
    RESULTS.append(res)
    return res


## 6. Query Bank
Test cases spanning negation, exception rules, keyword traps, and control queries.


In [11]:
QUERY_BANK = [
    {"key": "trade_negation", "cat": "Negation / constraint",
     "query": "Can two players who are not taking their turn trade resources with each other?",
     "gold": r"whose turn|may not trade|only trade|trade only|not .{0,25}turn.{0,60}trad",
     "expect": "Only the active player trades; tests handling of negative constraints."},

    {"key": "robber_stay", "cat": "Negation / constraint",
     "query": "Am I allowed to leave the robber on the same hex when I move it?",
     "gold": r"must (immediately )?move the robber|other (terrain )?hex|different (terrain )?hex|any other",
     "expect": "Tests whether 'must move' outranks general stealing clauses."},

    {"key": "win_other_turn", "cat": "Negation / constraint",
     "query": "I hit 10 victory points while another player was taking their turn, do I win immediately?",
     "gold": r"(10|ten).{0,50}victory points?.{0,80}(your|his|her|their) turn|(your|their) turn.{0,80}(10|ten) victory",
     "expect": "Winning condition requires 10 VP on active player turn."},

    {"key": "devcard_same_turn", "cat": "Exception / corner case",
     "query": "Can I play a development card right after buying it in the same turn?",
     "gold": r"(not|never|may not|cannot|can't).{0,90}(same|that) turn|except.{0,60}victory point",
     "expect": "Same-turn restriction with victory point exception."},

    {"key": "road_broken", "cat": "Exception / corner case",
     "query": "Does my longest road still count if an opponent builds a settlement in the middle of it?",
     "gold": r"(interrupt|broken|break|split).{0,140}(settlement|road)|(settlement|road).{0,140}(interrupt|broken|break)",
     "expect": "Road broken by opposing settlement; tests exception handling."},

    {"key": "road_tie", "cat": "Exception / corner case",
     "query": "Two players both have a road of the same length, who keeps the Longest Road card?",
     "gold": r"\btie\b|\btied\b|\bties\b",
     "expect": "Tie-break resolution rule."},

    {"key": "seven_discard", "cat": "Exception / corner case",
     "query": "I rolled a 7 and I only hold six cards, do I discard anything?",
     "gold": r"more than (7|seven)|half.{0,40}(round|down)",
     "expect": "Control query for 7-roll discard threshold."},

    {"key": "bank_trade_synonym", "cat": "Keyword trap / synonym",
     "query": "Can I swap my resource cards for something I need without involving other players?",
     "gold": r"4:1|four.{0,30}(same|identical)|maritime|harbor|3:1|2:1",
     "expect": "Tests maritime trade retrieval against lexical trade distractors."},

    {"key": "bigger_settlement", "cat": "Keyword trap / synonym",
     "query": "How do I make my settlement bigger and what extra do I get from it?",
     "gold": r"(city|cities).{0,120}(2|two) resource|(2|two) resource cards.{0,80}(city|cities)|replace.{0,60}settlement",
     "expect": "City upgrade rule; query mentions settlement rather than city."},

    {"key": "build_next_to", "cat": "Keyword trap / synonym",
     "query": "Can I put a new settlement right beside another player's settlement?",
     "gold": r"distance rule|adjacent intersections|3 adjacent|three adjacent|all 3 .{0,30}vacant",
     "expect": "Tests distance rule retrieval."},

    {"key": "second_settlement", "cat": "Buried chunk",
     "query": "During setup, which resources do I receive for my second settlement?",
     "gold": r"second settlement.{0,120}(resource|hex|terrain)|(resource|hex).{0,120}second settlement|reverse|opposite",
     "expect": "Tests setup phase rules spread across chunks."},

    {"key": "knight_army", "cat": "Buried chunk",
     "query": "How many knights do I have to play to take the Largest Army card?",
     "gold": r"(3|three).{0,40}knight|knight.{0,60}(3|three)|largest army.{0,80}(3|three)",
     "expect": "Control query with direct terminology match."},
]

BY_KEY = {q["key"]: q for q in QUERY_BANK}
print(len(QUERY_BANK), "queries in the bank")


12 queries in the bank


Verification of gold chunk matches across the corpus:


In [12]:
for q in QUERY_BANK:
    hits = find_chunks(q["gold"])
    print(f"{q['key']:<22} gold-matching chunks: {hits}")

trade_negation         gold-matching chunks: [9, 10, 12, 25, 41, 42, 53, 54, 55]
robber_stay            gold-matching chunks: [14, 15, 29, 35, 42, 43, 44, 45, 46]
win_other_turn         gold-matching chunks: [16, 17, 25, 26, 56, 70]
devcard_same_turn      gold-matching chunks: [15, 23]
road_broken            gold-matching chunks: [32, 33]
road_tie               gold-matching chunks: [33]
seven_discard          gold-matching chunks: [14, 43, 44]
bank_trade_synonym     gold-matching chunks: [0, 1, 7, 8, 9, 10, 21, 22, 28, 29, 33, 34, 35, 41, 42, 49, 50, 52, 53, 54, 61, 62, 68, 69, 71]
bigger_settlement      gold-matching chunks: [6, 9, 13, 20, 39]
build_next_to          gold-matching chunks: [7, 12, 24, 29, 45, 46, 48, 60]
second_settlement      gold-matching chunks: [48, 49]
knight_army            gold-matching chunks: [5, 6, 13, 15, 16, 22, 23, 30, 31, 58]


## 7. Query Screening (Retrieval Only)


In [13]:
def screen(bank):
    rows = []
    for q in bank:
        if q["key"] not in GOLD_IDS:
            continue
        cands = baseline_retrieve(q["query"])
        ranked = rerank(q["query"], cands)
        b = gold_rank([c for c, _ in cands], GOLD_IDS[q["key"]])
        r = gold_rank([c for c, _ in ranked], GOLD_IDS[q["key"]])
        if b is None:
            verdict = "not in top-10"
        elif b > 3 and r is not None and r <= 3:
            verdict = "RERANK WIN"
        elif b <= 3 and r <= 3:
            verdict = "both fine"
        elif b <= 3 and r > 3:
            verdict = "CE HURT"
        else:
            verdict = "both miss"
        rows.append({"key": q["key"], "category": q["cat"], "base_rank": b, "rerank_rank": r, "verdict": verdict})
    return pd.DataFrame(rows)

screen_df = screen(QUERY_BANK)
display(screen_df)
print(screen_df["verdict"].value_counts())


,key,category,base_rank,rerank_rank,verdict
0,trade_negation,Negation / constraint,8,1,RERANK WIN
1,win_other_turn,Negation / constraint,1,1,both fine
2,road_broken,Exception / corner case,5,2,RERANK WIN
3,bank_trade_synonym,Keyword trap / synonym,4,1,RERANK WIN
4,bigger_settlement,Keyword trap / synonym,7,2,RERANK WIN
5,knight_army,Buried chunk,2,1,both fine


verdict
RERANK WIN    4
both fine     2
Name: count, dtype: int64


## 8. Generation & End-to-End Comparison


In [ ]:
def run(key):
    q = BY_KEY[key]
    print(f"[{q['cat']}]  expectation: {q['expect']}")
    return compare(q["query"], GOLD_IDS.get(key), tag=key)

_ = run("trade_negation")


[Negation / constraint]  expectation: Baseline tends to return general trading chunks (which say players CAN trade). The restriction (only the active player trades) should be pulled up.
QUERY: Can two players who are not taking their turn trade resources with each other?


,chunk,page,base_rank,cosine,ce_score,rerank_rank,moved,gold,preview
0,10,4,8,0.435,6.83,1,7,YES,can also make their own proposals and counteroffers. Important: Players may only trad...
1,9,4,1,0.569,5.78,2,-1,YES,number rolled receives 1 resource card of the hex’s type. For an example see resource...
2,25,7,5,0.476,5.40,3,2,YES,"single or multiple cards. However, you cannot give away cards, or trade matching reso..."
3,8,4,9,0.414,3.02,4,5,,all players). • Y ou may trade Y resource cards with other players and/or use maritim...
4,41,10,2,0.512,1.85,5,-3,YES,"as many of these resources as are left in the supply, and any extras are lost. In eit..."
5,24,7,7,0.454,1.52,6,1,,all other VP cards) and win the game. You only reveal victory point cards when the ga...
6,34,9,4,0.484,1.36,7,-3,,trade resources using maritime trade during the trade phase even without involving an...
7,42,11,10,0.412,-0.30,8,2,YES,"among themselves, only with the player whose turn it is. There are 2 different kinds ..."
8,40,10,6,0.469,-0.65,9,-3,,"2 resource cards. Example: See Illustration J. Loren, the blue player, rolls a “4,” H..."
9,35,9,3,0.488,-4.03,10,-7,,"trade phase. Example: Brad has a settlement at a generic harbor. He can, for instance..."


GOLD chunk rank:  baseline #1  ->  re-ranked #1   (None = not in the top-10 at all)

ANSWER 1 (baseline):
 No, two players who are not taking their turn cannot trade resources with each other. Players may only trade with the player whose turn it is. The other players may not trade among themselves.

ANSWER 2 (re-ranked):
 No, two players who are not taking their turn may not trade resources with each other. Players may only trade with the player whose turn it is. The other players may not trade among themselves.


In [33]:
_ = run("devcard_same_turn")

[Exception / corner case]  expectation: Baseline returns the 'buying development cards' chunk; the same-turn restriction + Victory Point exception is elsewhere.
QUERY: Can I play a development card right after buying it in the same turn?


,chunk,page,base_rank,cosine,ce_score,rerank_rank,moved,gold,preview
0,23,7,1,0.581,8.11,1,0,YES,the other terrain types. development Cards There are 3 different kinds of development...
1,8,4,9,0.327,4.98,2,7,,all players). • Y ou may trade Y resource cards with other players and/or use maritim...
2,38,10,8,0.330,4.98,3,5,,"play this card, you may immediately place 2 free roads on the board (according to nor..."
3,55,14,3,0.391,2.46,4,-1,,• Y ou may not trade with the bank during another player’s turn. • Y ou may not give ...
4,19,6,2,0.411,0.82,5,-3,,"resource production and finished trading. To build, you must turn in the specified co..."
5,13,5,4,0.381,-1.60,6,-2,,is worth 1 victory point. c) City Y Requires: 3 Ore & 2 Grain You may only establish ...
6,11,4,7,0.338,-1.79,7,0,,"your road network, improve your resource production, and/or buy useful development ca..."
7,20,6,6,0.342,-2.50,8,-2,,Buildin G Costs Cards The building costs cards show what can be built and which resou...
8,18,5,5,0.343,-3.31,9,-4,,"hex. KNIGHT © 2015 Catan GmbH © 2015 Catan GmbH MONOPOLY When you play this card, ann..."
9,60,15,10,0.325,-4.67,10,0,,. . . . . . . . . . . . . . 6 Combined Trade/Build Phase . . . . . . . . . . . . . . ...


GOLD chunk rank:  baseline #1  ->  re-ranked #1   (None = not in the top-10 at all)

ANSWER 1 (baseline):
 No, you may not play a development card that you bought during the same turn. The only exception is if you buy a Victory Point card that brings you to 10 points; in this specific case, you may immediately reveal it (and all other VP cards) and win the game. Otherwise, development cards must be kept hidden until a later turn.

ANSWER 2 (re-ranked):
 No, you may not play a development card that you bought during the same turn. The only exception is if you buy a Victory Point card that brings you to 10 points, in which case you may immediately reveal it (and all other VP cards) to win the game.


In [34]:
_ = run("road_broken")

[Exception / corner case]  expectation: Generic Longest Road description ranks first; the 'broken by an opposing settlement' clause is buried.
QUERY: Does my longest road still count if an opponent builds a settlement in the middle of it?


,chunk,page,base_rank,cosine,ce_score,rerank_rank,moved,gold,preview
0,32,9,1,0.656,5.09,1,0,YES,"the first player to build a continuous road of at least 5 individual road pieces, you..."
1,33,9,5,0.543,3.55,2,3,YES,a settlement on an unoccupied intersection along that road! Example: In Illustration ...
2,12,4,2,0.629,0.63,3,-1,,"road than the one created by the current owner of the “Longest Road” card, they immed..."
3,42,11,4,0.551,0.45,4,0,,"among themselves, only with the player whose turn it is. There are 2 different kinds ..."
4,11,4,7,0.523,-1.11,5,2,,"your road network, improve your resource production, and/or buy useful development ca..."
5,26,7,8,0.512,-1.48,6,2,,"reach—10 victory points on your turn, the game ends immediately and you win! You can ..."
6,48,12,3,0.609,-2.02,7,-4,,begins. The starting player places a settlement on an unoccupied intersection Y of th...
7,46,11,6,0.541,-4.21,8,-2,,"distance rule, see Illustration M on page 12. Example: See Illustration L. It is Xand..."
8,45,11,9,0.509,-4.74,9,0,,settlement or city adjacent to this new hex. If there are 2 or more players with buil...
9,7,3,10,0.492,-5.92,10,0,,by the settlements marked with white stars. See HH. 44 Example: See Illustration A. B...


GOLD chunk rank:  baseline #1  ->  re-ranked #1   (None = not in the top-10 at all)

ANSWER 1 (baseline):
 No, if an opponent builds a settlement on an unoccupied intersection along your road, it breaks your road into two parts. The excerpt states that the player "must give the special card to" another player in this scenario. This means your original continuous road no longer counts as a single entity for the "Longest Road" card.

ANSWER 2 (re-ranked):
 No, an opponent building a settlement on an unoccupied intersection along your road breaks it into two parts. Your original continuous road no longer counts as a single longest road. You must then re-evaluate your longest continuous road. If you no longer have the longest road, you lose the "Longest Road" card, unless you are tied for the longest road after it's broken, in which case you keep the card.


In [35]:
_ = run("bank_trade_synonym")

[Keyword trap / synonym]  expectation: Heavy lexical overlap with domestic trade ('trade with players'), but the answer uses 'maritime trade' / 'harbor' / '4:1'.
QUERY: Can I swap my resource cards for something I need without involving other players?


,chunk,page,base_rank,cosine,ce_score,rerank_rank,moved,gold,preview
0,34,9,4,0.403,5.31,1,3,YES,trade resources using maritime trade during the trade phase even without involving an...
1,9,4,1,0.438,4.65,2,-1,YES,number rolled receives 1 resource card of the hex’s type. For an example see resource...
2,38,10,3,0.425,1.14,3,0,,"play this card, you may immediately place 2 free roads on the board (according to nor..."
3,30,8,6,0.372,0.08,4,2,,You then steal 1 resource card from a player who has a settlement or a city adjacent ...
4,44,11,8,0.355,-0.02,5,3,,"more than 7 resource cards (i.e., 8 or more) must choose and discard half of them. Re..."
5,40,10,2,0.430,-0.29,6,-4,,"2 resource cards. Example: See Illustration J. Loren, the blue player, rolls a “4,” H..."
6,24,7,9,0.349,-0.57,7,2,,all other VP cards) and win the game. You only reveal victory point cards when the ga...
7,35,9,7,0.362,-0.76,8,-1,YES,"trade phase. Example: Brad has a settlement at a generic harbor. He can, for instance..."
8,41,10,5,0.380,-1.55,9,-4,YES,"as many of these resources as are left in the supply, and any extras are lost. In eit..."
9,55,14,10,0.328,-1.84,10,0,,• Y ou may not trade with the bank during another player’s turn. • Y ou may not give ...


GOLD chunk rank:  baseline #1  ->  re-ranked #1   (None = not in the top-10 at all)

ANSWER 1 (baseline):
 Yes, you can swap your resource cards for something you need without involving other players through Maritime Trade. Excerpt 1 states, "You can also trade without the other players!"

ANSWER 2 (re-ranked):
 Yes, you can swap your resource cards for something you need without involving other players through Maritime Trade. The most basic exchange rate is 4 identical resource cards for 1 resource card of your choice, which does not require a harbor. If you have a settlement at a Generic Harbor, you can trade 3 identical resource cards for any 1 other resource card. There are also Special Harbors that allow a 2:1 exchange rate for a specific resource type if you have a settlement there.


In [36]:
_ = run("bigger_settlement")

[Keyword trap / synonym]  expectation: The word 'settlement' pulls in setup/distance-rule chunks, while the real answer talks about a 'city' and 2 resource cards.
QUERY: How do I make my settlement bigger and what extra do I get from it?


,chunk,page,base_rank,cosine,ce_score,rerank_rank,moved,gold,preview
0,71,16,2,0.470,-1.09,1,1,,you must build new roads and settlements and upgrade your settlements to cities. Each...
1,13,5,7,0.453,-1.77,2,5,YES,is worth 1 victory point. c) City Y Requires: 3 Ore & 2 Grain You may only establish ...
2,47,12,6,0.455,-2.45,3,3,,"settlement. You will then have the settlement in your supply, so you can build anothe..."
3,53,13,1,0.474,-2.65,4,-3,,"good forest or hills hex. • Do not underestimate the value of harbors. For instance, ..."
4,4,3,3,0.470,-3.10,5,-2,,"the Game Select a color and take your 5 settlements, 4 cities, and 15 roads (no more ..."
5,20,6,5,0.457,-3.13,6,-1,YES,Buildin G Costs Cards The building costs cards show what can be built and which resou...
6,45,11,9,0.441,-3.38,7,2,,settlement or city adjacent to this new hex. If there are 2 or more players with buil...
7,12,4,4,0.467,-4.55,8,-4,,"road than the one created by the current owner of the “Longest Road” card, they immed..."
8,70,16,10,0.434,-5.00,9,1,,"numbers. The taller the depicted number, and the more pips it has, the more likely th..."
9,52,13,8,0.442,-6.65,10,-2,,in Illustration R. • Place the terrain hexes exactly as shown in Illustration R. • Pl...


GOLD chunk rank:  baseline #5  ->  re-ranked #2   (None = not in the top-10 at all)

ANSWER 1 (baseline):
 You make your settlement bigger by upgrading it to a city. Each city is worth 2 victory points. To upgrade, you need to acquire resources.

ANSWER 2 (re-ranked):
 You make your settlement bigger by upgrading it to a city. You may only establish a city by upgrading one of your settlements, which requires 3 Ore and 2 Grain. When you upgrade, you replace the settlement piece with a city piece. Cities are worth 2 victory points and produce twice as many resources as settlements, meaning you acquire 2 resource cards for an adjacent terrain hex that produces resources.


In [37]:
_ = run("win_other_turn")

[Negation / constraint]  expectation: General 'ending the game' or VP-card chunks outrank the 'only on your own turn' condition.
QUERY: I hit 10 victory points while another player was taking their turn, do I win immediately?


,chunk,page,base_rank,cosine,ce_score,rerank_rank,moved,gold,preview
0,26,7,1,0.588,8.40,1,0,YES,"reach—10 victory points on your turn, the game ends immediately and you win! You can ..."
1,17,5,2,0.518,5.69,2,0,YES,"someone else wins. You may play any number of victory point cards during your turn, e..."
2,16,5,3,0.476,5.60,3,0,YES,remain face up in front of you. The first player to have 3 knight cards in front of t...
3,56,14,6,0.403,4.59,4,2,YES,"down in front of you. Of course, if you have 1 or 2 unused cards face down in front o..."
4,31,8,7,0.377,1.46,5,2,,"On Niall’s turn, he plays a knight card and moves the robber from the fields hex to t..."
5,32,9,8,0.349,1.01,6,2,,"the first player to build a continuous road of at least 5 individual road pieces, you..."
6,27,7,4,0.475,-0.01,7,-3,,the number of points required for victory. MARKET 2 Victory Points! This card goes to...
7,25,7,9,0.328,-1.02,8,1,YES,"single or multiple cards. However, you cannot give away cards, or trade matching reso..."
8,58,14,5,0.424,-1.50,9,-4,,"© 2015 Catan GmbH 1 Victory Point! Reveal this card on your turn if, with it, you rea..."
9,57,14,10,0.323,-5.11,10,0,,"VP cards, etc.). This is the VP symbol. It is a visual reminder for the things, other..."


GOLD chunk rank:  baseline #1  ->  re-ranked #1   (None = not in the top-10 at all)

(top-3 identical for both pipelines, so the answers are the same)

ANSWER 1 (baseline):
 No, you do not win immediately. You can only win during your turn. If you reach 10 victory points during another player's turn, you must wait until your next turn to claim victory. The game continues until any player (including you) has 10 points on their turn, and you may only reveal victory point cards during your turn to win.

ANSWER 2 (re-ranked):
 No, you do not win immediately. You can only win during your turn. If you reach 10 victory points during another player's turn, you must wait until your next turn to claim victory. The game continues until any player (including you) has 10 points on their turn, and you may only reveal victory point cards during your turn to win.


In [38]:
# a control query: shows what "nothing interesting happens" looks like
_ = run("knight_army")

[Buried chunk]  expectation: Likely a CONTROL: the answer uses the same words as the question.
QUERY: How many knights do I have to play to take the Largest Army card?


,chunk,page,base_rank,cosine,ce_score,rerank_rank,moved,gold,preview
0,16,5,2,0.555,5.97,1,1,YES,remain face up in front of you. The first player to have 3 knight cards in front of t...
1,31,8,1,0.582,5.68,2,-1,YES,"On Niall’s turn, he plays a knight card and moves the robber from the fields hex to t..."
2,30,8,3,0.554,4.35,3,0,YES,You then steal 1 resource card from a player who has a settlement or a city adjacent ...
3,61,15,9,0.383,0.22,4,5,,. 7 Ending the Game . . . . . . . . . . . . . . . . . . . . . . 7 Game Play. . . . . ...
4,15,5,7,0.442,-0.13,5,2,YES,then take 1 card at random. If the target hex is adjacent to 2 or more players’ settl...
5,56,14,8,0.394,-0.37,6,2,,"down in front of you. Of course, if you have 1 or 2 unused cards face down in front o..."
6,59,14,5,0.476,-0.63,7,-2,,who plays more Knight cards takes this card. 2 Victory Points! This card goes to the ...
7,58,14,4,0.539,-2.49,8,-4,YES,"© 2015 Catan GmbH 1 Victory Point! Reveal this card on your turn if, with it, you rea..."
8,51,13,6,0.451,-8.67,9,-3,,"the desert. Important: Alternatively, you can use a fully random set-up. Place 1 toke..."
9,27,7,10,0.382,-9.91,10,0,,the number of points required for victory. MARKET 2 Victory Points! This card goes to...


GOLD chunk rank:  baseline #1  ->  re-ranked #1   (None = not in the top-10 at all)
  gemini error (ClientError): retrying in 5s
  gemini error (ClientError): retrying in 10s

ANSWER 1 (baseline):
 To initially receive the Largest Army card, you must be the first player to play 3 knight cards. If another player already holds the card, you must play more knight cards than they currently have to take it.

ANSWER 2 (re-ranked):
 You must be the first player to have 3 knight cards face up in front of you to receive the Largest Army card. If another player already holds the card, you must have more face-up knight cards than them to take it.


### Try my own query
Edit the string and run. If I know the answer sits in a chunk with a particular phrase, I can pass it as `gold_regex`.

In [39]:
my_query = "If a player has 15 cards which include 2 development card and 13 resource card and 7 is rolled on dice by another player, how many cards should the player forgo?"
_ = compare(my_query, gold_regex=None)

QUERY: If a player has 15 cards which include 2 development card and 13 resource card and 7 is rolled on dice by another player, how many cards should the player forgo?


,chunk,page,base_rank,cosine,ce_score,rerank_rank,moved,gold,preview
0,44,11,2,0.439,1.91,1,1,,"more than 7 resource cards (i.e., 8 or more) must choose and discard half of them. Re..."
1,38,10,5,0.414,0.71,2,3,,"play this card, you may immediately place 2 free roads on the board (according to nor..."
2,8,4,1,0.454,0.69,3,-2,,all players). • Y ou may trade Y resource cards with other players and/or use maritim...
3,47,12,9,0.396,-0.10,4,5,,"settlement. You will then have the settlement in your supply, so you can build anothe..."
4,23,7,3,0.432,-0.28,5,-2,,the other terrain types. development Cards There are 3 different kinds of development...
5,4,3,10,0.387,-1.21,6,4,,"the Game Select a color and take your 5 settlements, 4 cities, and 15 roads (no more ..."
6,39,10,6,0.411,-1.97,7,-1,,"lumber (from forest), and wool (from pasture). These resources are represented by “re..."
7,60,15,4,0.427,-2.75,8,-4,,. . . . . . . . . . . . . . 6 Combined Trade/Build Phase . . . . . . . . . . . . . . ...
8,20,6,8,0.399,-2.86,9,-1,,Buildin G Costs Cards The building costs cards show what can be built and which resou...
9,18,5,7,0.399,-6.33,10,-3,,"hex. KNIGHT © 2015 Catan GmbH © 2015 Catan GmbH MONOPOLY When you play this card, ann..."



ANSWER 1 (baseline):
 When a "7" is rolled, a player with more than 7 resource cards must discard half of them. The player in question has 13 resource cards. Since 13 is an odd number, the player must round down when discarding half. Therefore, the player must discard 6 resource cards (13 / 2 = 6.5, rounded down to 6). Development cards are not subject to this rule, as the excerpt specifies "resource cards."

ANSWER 2 (re-ranked):
 When a 7 is rolled, a player with more than 7 resource cards must discard half of them, rounded down. In this case, the player has 13 resource cards. Half of 13 is 6.5, which rounds down to 6. Therefore, the player should discard 6 resource cards. Development cards are not counted towards this limit.


## 9. Rank-shift summary of the full runs

In [40]:
summary = pd.DataFrame([{
    "query_key": r["tag"],
    "gold_rank_baseline": r["gold_base"],
    "gold_rank_reranked": r["gold_rerank"],
} for r in RESULTS if r["tag"]])
display(summary)

,query_key,gold_rank_baseline,gold_rank_reranked
0,trade_negation,1,1
1,devcard_same_turn,1,1
2,road_broken,1,1
3,bank_trade_synonym,1,1
4,bigger_settlement,5,2
5,win_other_turn,1,1
6,knight_army,1,1


## 10. Latency profiling

I average over several repeats after a warm-up run (first calls are slower because of lazy loading / caching). Gemini time is measured separately because it's network-bound and has nothing to do with retrieval quality.

In [41]:
LATENCY_KEYS = ["trade_negation", "devcard_same_turn", "road_broken", "bank_trade_synonym", "bigger_settlement"]
REPS = 5

# warm-up
q0 = BY_KEY[LATENCY_KEYS[0]]["query"]
_c = baseline_retrieve(q0); rerank(q0, _c)

rows = []
for key in LATENCY_KEYS:
    q = BY_KEY[key]["query"]
    embed_t, search_t, ce_t = [], [], []
    for _ in range(REPS):
        t0 = time.perf_counter(); qv = embed_query(q);             embed_t.append(time.perf_counter() - t0)
        t0 = time.perf_counter(); s, ids = index.search(qv, TOP_K_RETRIEVE); search_t.append(time.perf_counter() - t0)
        cands = [(int(i), float(sc)) for i, sc in zip(ids[0], s[0])]
        t0 = time.perf_counter(); rerank(q, cands);                ce_t.append(time.perf_counter() - t0)
    rows.append({
        "query": key,
        "embed_query_ms": 1000 * np.mean(embed_t),
        "faiss_search_ms": 1000 * np.mean(search_t),
        "pipeline1_retrieval_ms": 1000 * (np.mean(embed_t) + np.mean(search_t)),
        "cross_encoder_10_pairs_ms": 1000 * np.mean(ce_t),
        "pipeline2_retrieval_ms": 1000 * (np.mean(embed_t) + np.mean(search_t) + np.mean(ce_t)),
    })

lat = pd.DataFrame(rows).set_index("query")
lat.loc["MEAN"] = lat.mean()
display(lat.round(2))

llm_times = [r["t_llm1"] for r in RESULTS if r["t_llm1"] == r["t_llm1"]]   # drops NaN
if llm_times:
    print(f"\nGemini generation ({GEN_MODEL}): mean {np.mean(llm_times):.2f}s over {len(llm_times)} calls")
print(f"One-time cost of embedding + indexing {len(chunks)} chunks: {index_build_time:.2f}s")
print("device used by cross-encoder:", cross_encoder.model.device)

,embed_query_ms,faiss_search_ms,pipeline1_retrieval_ms,cross_encoder_10_pairs_ms,pipeline2_retrieval_ms
query,,,,,
trade_negation,5.62,0.03,5.65,59.27,64.91
devcard_same_turn,6.06,0.02,6.08,55.28,61.36
road_broken,6.00,0.03,6.03,60.66,66.68
bank_trade_synonym,6.68,0.03,6.71,62.60,69.31
bigger_settlement,5.02,0.02,5.04,94.36,99.39
MEAN,5.87,0.03,5.90,66.43,72.33



Gemini generation (gemini-2.5-flash): mean 4.67s over 8 calls
One-time cost of embedding + indexing 73 chunks: 0.45s
device used by cross-encoder: mps:0


## 11. Scaling experiment: why not cross-encode everything?

Two measurements:
1. Time the cross-encoder over **all** chunks of this (tiny) document and extrapolate the per-pair cost.
2. Time a FAISS search over a bigger random index, to see how cheap vector search is by comparison.

In [42]:
q = BY_KEY["trade_negation"]["query"]

all_pairs = [(q, c["text"]) for c in chunks]
t0 = time.perf_counter(); cross_encoder.predict(all_pairs); t_all = time.perf_counter() - t0
per_pair = t_all / len(all_pairs)
print(f"cross-encoder over ALL {len(chunks)} chunks: {t_all:.2f}s  ({1000 * per_pair:.1f} ms per pair)\n")

proj = pd.DataFrame({"collection_size": [1_000, 10_000, 100_000, 1_000_000]})
proj["est_cross_encoder_time_per_query"] = proj["collection_size"].apply(
    lambda n: f"{per_pair * n:,.0f} s  (~{per_pair * n / 60:,.1f} min)")
display(proj)

cross-encoder over ALL 73 chunks: 1.24s  (17.0 ms per pair)



,collection_size,est_cross_encoder_time_per_query
0,1000,17 s (~0.3 min)
1,10000,170 s (~2.8 min)
2,100000,"1,700 s (~28.3 min)"
3,1000000,"16,998 s (~283.3 min)"


In [43]:
# FAISS on a much bigger (random) collection, just to see the order of magnitude. Flat search is still linear
# but each comparison is a dot product, and ANN indexes (HNSW/IVF) go sub-linear.
N_BIG = 100_000
big = np.random.randn(N_BIG, doc_vecs.shape[1]).astype("float32")
big /= np.linalg.norm(big, axis=1, keepdims=True)
big_index = faiss.IndexFlatIP(big.shape[1]); big_index.add(big)

qv = embed_query(q)
t0 = time.perf_counter(); big_index.search(qv, 10); t_flat = time.perf_counter() - t0
print(f"FAISS flat search over {N_BIG:,} vectors: {1000 * t_flat:.1f} ms")
print(f"Cross-encoder over {N_BIG:,} chunks would take ~{per_pair * N_BIG:,.0f} s on this machine")
del big, big_index

FAISS flat search over 100,000 vectors: 6.2 ms
Cross-encoder over 100,000 chunks would take ~1,700 s on this machine


## 12. My observations 

### 1. Rank shifts (rank of the chunk that actually contains the answer)

| Query | Category | Answer chunk | Baseline rank | Re-ranked rank | Effect |
|---|---|---|:---:|:---:|---|
| bigger_settlement | Keyword trap | 13 (City: 3 ore and 2 grain) | 7 | 2 | Promoted into top 3. "settlement" pulled in setup and tactics chunks. |
| road_broken | Exception | 33 (opponent settlement breaks the road) | 5 | 2 | Promoted into top 3. |
| bank_trade_synonym | Keyword trap / synonym | 34 (maritime trade) | 4 | 1 | Promoted to #1. |
| trade_negation | Negation | 10 (only the active player trades) | 8 | 1 | Ranking improved, but the answer did not change (see below). Chunk 42 moved 10 to 8. |
| win_other_turn | Control | 26 | 1 | 1 | No change. |
| knight_army | Control | 16 | 2 | 1 | Same top 3 set, no change in answer. |

Note: my first gold regexes matched too many chunks, so the automatic gold rank was unreliable. The ranks above were taken by reading the retrieved text.

### 2. Answer 1 (baseline) vs Answer 2 (re-ranked)

- bigger_settlement: Answer 1 was an omission (no cost, no production). Answer 2 gave 3 ore and 2 grain and 2 resource cards.
- bank_trade_synonym: Answer 1 was correct but vague (no 4:1, 3:1 or 2:1 rates). Answer 2 gave all the rates.
- road_broken: Answer 1 misread the rule about a longer road taking the card ("must give the special card") and did not mention a recount. Answer 2 gave the split, the recount and the tie case. Its tie line matches the rulebook special case (tied after a break, you keep the card). It does not mention the other case, where the card is set aside.
- trade_negation: both answers were correct. Chunk 9 (baseline rank 1) ends with the same sentence that starts chunk 10, because of the 40 word overlap. So the rule reached Gemini even though chunk 10 was at rank 8. Overlap can hide a ranking problem.

### 3. Latency (mean of 5 runs x 5 queries, cross-encoder on mps)

| Step | Baseline | With re-ranking |
|---|:---:|:---:|
| Query embedding | 5.87 ms | 5.87 ms |
| FAISS search | 0.03 ms | 0.03 ms |
| Cross-encoder, 10 pairs | not used | 66.43 ms |
| Total retrieval | 5.90 ms | 72.33 ms |
| Gemini (gemini-2.5-flash, mean of 8 calls) | 4.67 s | 4.67 s |

Re-ranking makes retrieval about 12x slower but it is under 2 percent of the end to end time.

### 4. Scaling

Cross-encoding all 73 chunks took 1.24 s, about 17.0 ms per pair. Projected per query: 1,000 chunks about 17 s, 10,000 about 2.8 min, 100,000 about 28 min, 1,000,000 about 4.7 hours. FAISS flat search over 100,000 random vectors took 6.2 ms, roughly 270,000 times faster. (The 10 pair run suggests about 6.6 ms per pair, which still gives about 1.8 hours for 1M chunks.) Cross-encoder scores depend on the query, so nothing can be precomputed, and cost grows linearly with the collection size. This is why we use the bi-encoder first and re-rank only the top 10.